Unstructured Data Project: Crime at ND

Joe Brady

Feb 2026

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#packages
!pip install tabula-py
import tabula
import pandas as pd
import os
import re

In [8]:
#get non-2026 data

#find year folders; just 2025 for now
folder_name = '/content/drive/MyDrive/NDPD_Blotter/'
all_files = os.listdir(folder_name)
folders = [folder for folder in all_files if len(folder) == 4 and int(folder) >= 2020]

#initialize dataframe
crime_df = pd.DataFrame([])
problem_files = []

#get the data
for folder in folders:
  folder_path = '/content/drive/MyDrive/NDPD_Blotter/' + folder
  all_files = os.listdir(folder_path)
  #all_files = [file for file in all_files if int(file[:2]) >=8]

  for file in all_files:
    if 'pdf' in file:
      print('\r'+file, end="")
      temp = tabula.read_pdf(folder_path + '/' + file,
                            pages='all',
                            lattice = True,
                            guess = False,
                            pandas_options={'header': None})
      #check for bad files
      if min([len(df.columns) for df in temp]) < 7:
        problem_files.append(file)
      #read and write into dataframe
      else:
        temp[0] = temp[0].iloc[1:]
        for i in range(0, len(temp)):
          temp[i] = temp[i].iloc[:, :7]
          temp[i].columns = ['incident', 'start_dt', 'end_dt', 'reported_dt', 'offence', 'address', 'disposition']
        df_f = pd.concat(temp)
        #save into mega df
        crime_df = pd.concat([crime_df, df_f], axis=0, ignore_index=True)

#view results
crime_df.head()


122225.pdf

,incident,start_dt,end_dt,reported_dt,offence,address,disposition
0,22‐002313,11:53:26 12/19/22,11:56:14 12/28/22,11:53:31 12/28/22,HARASSMENT,54140 S R 933; FATIMA; Our Lady of Fatima,Pending
1,23‐000007,01:04:25 01/04/23,01:04:25 01/04/23,01:04:25 01/04/23,MOTOR VEHICLE‐DRIVING WHILE\rSUSPENDED‐ PRIOR,DOUGLAS RD & JUNIPER RD,Arrest
2,22‐002271,16:30:00 12/12/22,22:30:00 12/12/22,15:55:21 12/14/22,THEFT,19050 MOOSE KRAUSE NORTH; DEBART,HR Review
3,22‐002274,13:00:00 12/13/22,20:00:00 12/14/22,11:48:15 12/15/22,THEFT,18940 LIBRARY LN; LIBR; Hesburgh Library,HR Review
4,23‐000009,13:45:11 01/05/23,14:51:28 01/05/23,13:45:13 01/05/23,CRIMINAL MISCHIEF,1 LAKE LOT; LAKEL; Lake Lot,Pending


In [30]:
#handle our 1 problem file, 1/2/2025 that has no lattice in formatting
filename =  '/content/drive/MyDrive/NDPD_Blotter/' + '2025/' + '010225.pdf' #manually set
temp1 = tabula.read_pdf(filename,
                            pages=1,
                            lattice = True,
                            guess = False,
                            pandas_options={'header': None})

temp2 = tabula.read_pdf(filename,
                            pages='2-3',
                            stream = True,
                            guess = False,
                            pandas_options={'header': None})

#combine to reuse code from above
temp = temp1 + temp2

#same processing
temp[0] = temp[0].iloc[1:]
for i in range(0, len(temp)):
  temp[i] = temp[i].iloc[:, :7]
  temp[i].columns = ['incident', 'start_dt', 'end_dt', 'reported_dt', 'offence', 'address', 'disposition']
problem_df = pd.concat(temp)

problem_df.head()


,incident,start_dt,end_dt,reported_dt,offence,address,disposition
30,OCS,00:00:00 10/18/24,00:00:00 10/19/24,09:00:00 12/20/24,LIQUOR VIOLATION,Zahm Hall,OCS Review
31,OCS,00:00:00 10/18/24,00:00:00 10/19/24,09:00:00 12/20/24,LIQUOR VIOLATION,Zahm Hall,OCS Review
32,OCS,00:00:00 10/18/24,00:00:00 10/19/24,09:00:00 12/20/24,LIQUOR VIOLATION,Zahm Hall,OCS Review
33,OCS,00:00:00 10/18/24,00:00:00 10/19/24,09:00:00 12/20/24,LIQUOR VIOLATION,Zahm Hall,OCS Review
34,OCS,00:00:00 10/18/24,00:00:00 10/19/24,09:00:00 12/20/24,LIQUOR VIOLATION,Zahm Hall,OCS Review


In [11]:
#get 2026 data

folder_name = '/content/drive/MyDrive/NDPD_Blotter'
all_files = os.listdir(folder_name)

#filter for only the 2025 academic school year, starting in august
rel_files = [file for file in all_files if 'pdf' in file]

#initialize dataframe
crime_2026_df = pd.DataFrame([])

#get it
for file in rel_files:
  if 'pdf' in file:
      print('\r'+file, end="")
      temp = tabula.read_pdf(folder_name + '/' + file,
                            pages='all',
                            lattice = True,
                            guess = False,
                            pandas_options={'header': None})
      #check for bad files
      if min([len(df.columns) for df in temp]) < 7:
        problem_files.append(file)
      #read and write into dataframe
      else:
        temp[0] = temp[0].iloc[1:]
        for i in range(0, len(temp)):
          temp[i] = temp[i].iloc[:, :7]
          temp[i].columns = ['incident', 'start_dt', 'end_dt', 'reported_dt', 'offence', 'address', 'disposition']
        df_f = pd.concat(temp)
        #save files to final df
        crime_2026_df = pd.concat([crime_2026_df, df_f], axis = 0)

crime_2026_df.head()

022726.pdf

,incident,start_dt,end_dt,reported_dt,offence,address,disposition
1,26-000007,23:53:40 01/03/26,00:07:40 01/04/26,23:53:40 01/03/26,DRUG VIOLATION,DOUGLAS RD & SR 933 HWY,Exceptional
2,26-000005,19:21:41 01/03/26,19:30:37 01/03/26,19:21:45 01/03/26,TRESPASS,54725 Notre Dame Ave; Morris Inn; Morris Inn,Arrest
3,25-001542,05:23:10 12/28/25,05:30:02 12/28/25,05:23:17 12/28/25,TRESPASS,54905 Bookstore Dr. Dr; Baumer Hall; Baumer Hall,Pending
4,25-001540,10:59:13 12/27/25,10:59:13 12/27/25,10:59:13 12/27/25,MOTOR VEHICLE- LEAVING THE SCENE OF A\rPROPERT...,18301 Douglas Rd Rd; Warren Golf Course; Warre...,Pending
5,25-001539,00:26:49 12/27/25,00:26:49 12/27/25,00:26:49 12/27/25,MOTOR VEHICLE-OPERATING A VEHICLE WHILE\rINTOX...,DOUGLAS RD & JUNIPER RD,Arrest


In [35]:
#concatenate all
final_df = pd.concat([crime_df, crime_2026_df, problem_df]).reset_index(drop=True)

final_df.head()
final_df.shape


(7385, 7)

In [34]:
#write file to use is VS code
final_df.to_csv('/content/drive/Shareddrives/MSBA Mod 3/Unstructured Data/Project/crime_2020to2026_all.csv')